In [1]:
from pathlib import Path
from pypdf import PdfReader

PDF_DIR = Path("data/pdfs")
pages = []

for pdf in PDF_DIR.glob("*.pdf"):
    reader = PdfReader(pdf)
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"doc": pdf.name, "page": i, "text": text})

print("Total pages with text:", len(pages))
print(pages[0]["doc"], pages[0]["page"], pages[0]["text"][:300])

Total pages with text: 30
CS201_Data_Structures_and_Algorithms.pdf 1 CS201
Page 1
CS201: Data Structures and Algorithms
Chapter 1: Algorithm Analysis and Big-O
An algorithm is a finite sequence of steps that solves a problem. To compare algorithms we
measure how their running time and memory use grow as the input size n grows. Big-O
notation describes the upper bound


In [2]:
def chunk_text(text, size=800, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks = []
for p in pages:
    for j, c in enumerate(chunk_text(p["text"])):
        chunks.append({
            "id": f'{p["doc"]}-p{p["page"]}-c{j}',
            "text": c, "doc": p["doc"], "page": p["page"],
        })
print("Total chunks:", len(chunks))

Total chunks: 60


In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../chroma_db")
col = client.get_or_create_collection("course", metadata={"hnsw:space": "cosine"})

embeddings = model.encode([c["text"] for c in chunks], show_progress_bar=True).tolist()

col.upsert(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=embeddings,
    metadatas=[{"doc": c["doc"], "page": c["page"]} for c in chunks],
)
print("Stored:", col.count())

W1107 11:28:07.688000 16111 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Stored: 60


In [4]:
def retrieve(question, k=4):
    q_emb = model.encode([question]).tolist()
    r = col.query(query_embeddings=q_emb, n_results=k)
    return [
        {"text": d, "doc": m["doc"], "page": m["page"], "score": round(1 - dist, 3)}
        for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])
    ]

# 1) course ka sawal
for r in retrieve("YAHAN apni course ka koi sawal likho"):
    print(r["score"], r["doc"], "p.", r["page"], "|", r["text"][:100].replace("\n", " "))

print("-----")
# 2) course se bahar ka sawal
for r in retrieve("Who won the FIFA World Cup in 2018?"):
    print(r["score"], r["doc"], "p.", r["page"], "|", r["text"][:100].replace("\n", " "))

0.188 CS101_Python_Programming.pdf p. 2 | k statement stops the loop immediately, while the continue statement skips the rest of the current i
0.181 CS201_Data_Structures_and_Algorithms.pdf p. 3 | ace at the start of the array to avoid wasting memory. A priority queue is a special queue where eac
0.143 CS401_Machine_Learning_Basics.pdf p. 2 | ient descent is an optimisation algorithm that does this step by step. At each step it computes the 
0.13 CS201_Data_Structures_and_Algorithms.pdf p. 4 | es skewed like a linked list. Balanced trees such as AVL trees and red-black trees keep the height a
-----
0.113 CS401_Machine_Learning_Basics.pdf p. 3 | ction of predicted positives that are truly positive: TP / (TP + FP). Recall is the fraction of actu
0.091 CS401_Machine_Learning_Basics.pdf p. 3 | CS401 Page 3 CS401: Machine Learning Basics Chapter 3: Classification and Evaluation Metrics Classif
0.059 CS101_Python_Programming.pdf p. 5 |  runs. The optional finally block always runs, whether o

In [5]:
for r in retrieve("Describe the TCP three-way handshake"):
    print(r["score"], r["doc"], "p.", r["page"], "|", r["text"][:100].replace("\n", " "))

0.624 CS302_Computer_Networks.pdf p. 3 | CS302 Page 3 CS302: Computer Networks Chapter 3: Transport Layer: TCP and UDP The transport layer de
0.491 CS302_Computer_Networks.pdf p. 1 | CS302 Page 1 CS302: Computer Networks Chapter 1: The OSI and TCP/IP Models A network model divides c
0.39 CS302_Computer_Networks.pdf p. 3 | onnectionless and unreliable: it sends datagrams without a handshake, without acknowledgements and w
0.358 CS302_Computer_Networks.pdf p. 1 | layers handle sessions, data formatting and encryption, and user-facing protocols such as HTTP and S


In [6]:
import csv
from pathlib import Path

csv_path = next(p for p in [Path("data/test_questions.csv"), Path("../data/test_questions.csv")] if p.exists())
rows = list(csv.DictReader(open(csv_path, encoding="utf-8")))

in_scores, out_scores, hits = [], [], 0
for r in rows:
    res = retrieve(r["question"], k=4)
    top = res[0]["score"]
    if r["expected_doc"]:
        in_scores.append(top)
        if any(x["doc"] == r["expected_doc"] and x["page"] == int(r["expected_page"]) for x in res):
            hits += 1
        else:
            print("MISS:", r["question"], "->", [(x["doc"][:5], x["page"]) for x in res])
    else:
        out_scores.append(top)

print(f"Sahi page top-4 mein: {hits}/{len(in_scores)}")
print("In-course  top score  -> min:", min(in_scores), " avg:", round(sum(in_scores)/len(in_scores), 3))
print("Out-course top score  -> max:", max(out_scores), " avg:", round(sum(out_scores)/len(out_scores), 3))

Sahi page top-4 mein: 20/20
In-course  top score  -> min: 0.267  avg: 0.534
Out-course top score  -> max: 0.355  avg: 0.19


In [7]:
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os, re, getpass
from openai import OpenAI

# --- provider (default: Groq) ---
BASE_URL ="https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL    = "gemini-3.5-flash-lite"
# Gemini ke liye: BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/" ; MODEL = "gemini-2.5-flash"
# Ollama ke liye: BASE_URL = "http://localhost:11434/v1" ; MODEL = "llama3.2"  (key mein kuch bhi likh do)
# Model ke naam kabhi kabhi badal jate hain, provider ke console mein current naam check kar lena.

api_key = os.environ.get("LLM_API_KEY") or getpass.getpass("API key: ")
llm = OpenAI(base_url=BASE_URL, api_key=api_key)

In [9]:
MIN_SCORE = 0.25
NOT_COVERED_MSG = "Sorry, this question is not covered in the course material I have."

SYSTEM = (
    "You are a course assistant for students. Answer ONLY using the numbered context excerpts provided. "
    "Do not use outside knowledge. Add the excerpt number in square brackets, like [1], after each fact you use. "
    "If the excerpts do not contain the answer, reply with exactly: NOT_COVERED"
)

def ask(question, k=4):
    hits = retrieve(question, k)
    if not hits or hits[0]["score"] < MIN_SCORE:
        return {"answer": NOT_COVERED_MSG, "sources": [], "covered": False}

    context = "\n\n".join(
        f"[{i+1}] ({h['doc']}, page {h['page']})\n{h['text']}" for i, h in enumerate(hits)
    )
    resp = llm.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    text = resp.choices[0].message.content.strip()
    if "NOT_COVERED" in text:
        return {"answer": NOT_COVERED_MSG, "sources": [], "covered": False}

    used = sorted({int(n) for n in re.findall(r"\[(\d+)\]", text) if 1 <= int(n) <= len(hits)})
    cited = [hits[n - 1] for n in used] or hits[:1]
    seen, sources = set(), []
    for h in cited:
        if (h["doc"], h["page"]) not in seen:
            seen.add((h["doc"], h["page"]))
            sources.append({"doc": h["doc"], "page": h["page"]})
    return {"answer": text, "sources": sources, "covered": True}

In [10]:
print(ask("Describe the TCP three-way handshake"))
print(ask("Who won the FIFA World Cup in 2018?"))

{'answer': 'The TCP three-way handshake uses the steps SYN, SYN-ACK, and ACK to set up a connection [1].', 'sources': [{'doc': 'CS302_Computer_Networks.pdf', 'page': 3}], 'covered': True}
{'answer': 'Sorry, this question is not covered in the course material I have.', 'sources': [], 'covered': False}


In [11]:
import time
ok_cov = ok_page = 0
n_in = sum(1 for r in rows if r["expected_doc"])

for r in rows:
    out = ask(r["question"])
    should = bool(r["expected_doc"])
    if out["covered"] == should:
        ok_cov += 1
    else:
        print("COVERAGE GALAT:", r["question"], "-> covered =", out["covered"])
    if should and any(s["doc"] == r["expected_doc"] and s["page"] == int(r["expected_page"]) for s in out["sources"]):
        ok_page += 1
    time.sleep(1)

print(f"Covered/not-covered sahi: {ok_cov}/{len(rows)}")
print(f"Citation mein sahi page : {ok_page}/{n_in}")

Covered/not-covered sahi: 26/26
Citation mein sahi page : 20/20
